<a href="https://colab.research.google.com/github/shruthilakshmi008-BA/ba-automation-suite/blob/main/03.%20User-Story-Generator/03_User_Story_Generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q google-genai pandas

In [2]:
import os
import re
import json
import pandas as pd

from google import genai
from google.colab import files, userdata

In [5]:
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

if GEMINI_API_KEY:
    print("Gemini API key loaded successfully.")
else:
    print("Gemini API key not found.")

client = genai.Client(api_key=GEMINI_API_KEY)

print("Gemini client initialized successfully!")

Gemini API key loaded successfully.
Gemini client initialized successfully!


In [6]:
uploaded = files.upload()
input_file = list(uploaded.keys())[0]
print(f"Input file selected: {input_file}")

df = pd.read_csv(input_file)

print("Input file loaded successfully.")
print(f"Number of requirements: {len(df)}")

df.head()

Saving requirements_input.csv to requirements_input.csv
Input file selected: requirements_input.csv
Input file loaded successfully.
Number of requirements: 5


,requirement_id,description,type,priority
0,REQ-001,The biggest issue right now is that when a pur...,Functional,High
1,REQ-002,People keep submitting requests without attach...,Functional,Medium
2,REQ-003,The system does not send any reminder if an ap...,Non-Functional,High
3,REQ-004,Need approvals to happen within 3 business day...,Non-Functional,High
4,REQ-005,Finance should see a simple dashboard of all p...,Functional,Low


In [7]:
required_columns = [
    "requirement_id",
    "description"
]

missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

print("Input CSV structure is valid.")

Input CSV structure is valid.


In [8]:
##Fallback Mechanism
ROLE_KEYWORDS = {
    "approver": ["approve", "approval"],
    "finance team member": ["finance"],
    "branch user": ["branch"],
    "procurement officer": [
        "vendor",
        "quote",
        "procurement"
    ],
}


def infer_role(description: str) -> str:
    lowered = str(description).lower()

    for role, keywords in ROLE_KEYWORDS.items():
        if any(k in lowered for k in keywords):
            return role

    return "system user"

def mock_draft_user_story(row) -> str:
    role = infer_role(row["description"])

    goal = (
        str(row["description"])
        .rstrip(".")
        .strip()
    )

    return (
        f"As a {role}, "
        f"I want to ensure that {goal.lower()}, "
        f"so that the process is efficient and auditable."
    )

def mock_draft_acceptance_criteria(row) -> str:

    req_type = str(
        row.get("type", "Functional")
    )

    priority = str(
        row.get("priority", "Medium")
    )

    if req_type == "Non-Functional":
        return (
            "Given the system is running operational workflows, "
            "When threshold limits or SLA deadlines are reached, "
            "Then the system enforces constraints automatically "
            f"(Priority: {priority})."
        )

    return (
        "Given an authorized user is in the portal, "
        f"When they execute the action matching "
        f"'{row['requirement_id']}', "
        "Then the application completes the request "
        f"as specified (Priority: {priority})."
    )

In [9]:
##AI Generator
def generate_ai_stories(df: pd.DataFrame) -> pd.DataFrame:

    user_stories = []
    acceptance_criteria_list = []

    print(
        "[AI Pipeline] Generating User Stories "
        "and Acceptance Criteria using Gemini..."
    )

    for index, row in df.iterrows():

        requirement_id = row["requirement_id"]
        description = row["description"]
        req_type = row.get("type", "Functional")
        priority = row.get("priority", "Medium")

        prompt = f"""
You are a Senior Technical Business Analyst
and Agile Product Owner.

Convert the following requirement into a high-quality
Agile User Story and Acceptance Criteria.

Requirement ID:
{requirement_id}

Requirement Description:
{description}

Requirement Type:
{req_type}

Priority:
{priority}

Return ONLY a valid JSON object.

The JSON must contain exactly two keys:

"user_story"

and

"acceptance_criteria"

User Story requirements:

Use this exact structure:

As a [specific role],
I want [specific goal],
so that [business benefit].

The role must be appropriate for the requirement.

Acceptance Criteria requirements:

Provide 2 to 3 specific Given/When/Then statements.

They must be directly related to the requirement.

Format acceptance criteria as a numbered list inside
the JSON string.

Example:

{{
    "user_story": "As a finance team member, I want to review payment requests, so that I can ensure payments are accurate and compliant.",
    "acceptance_criteria": "1. Given a payment request exists, When the finance user opens it, Then the request details are displayed.\\n2. Given the request is valid, When the finance user approves it, Then the request status changes to Approved."
}}

Do not return markdown.
Do not return ```json.
Do not include explanations outside the JSON.
"""

        try:

            response = client.models.generate_content(
                model="gemini-3.6-flash",
                contents=prompt
            )

            raw_text = response.text.strip()

            # Remove markdown code fences if Gemini adds them
            raw_text = re.sub(
                r"^```json\s*|\s*```$",
                "",
                raw_text,
                flags=re.IGNORECASE
            ).strip()

            result = json.loads(raw_text)

            user_story = result.get(
                "user_story",
                mock_draft_user_story(row)
            )

            acceptance_criteria = result.get(
                "acceptance_criteria",
                mock_draft_acceptance_criteria(row)
            )

        except Exception as e:

            print(
                f"Warning: Gemini failed for "
                f"{requirement_id}: {e}"
            )

            user_story = mock_draft_user_story(row)

            acceptance_criteria = (
                mock_draft_acceptance_criteria(row)
            )

        user_stories.append(user_story)
        acceptance_criteria_list.append(
            acceptance_criteria
        )

        print(
            f"Processed {index + 1}/{len(df)}: "
            f"{requirement_id}"
        )

    df["user_story"] = user_stories

    df["acceptance_criteria"] = (
        acceptance_criteria_list
    )

    df["generation_method"] = "Google Gemini"

    return df

In [10]:
df_result = generate_ai_stories(df)

print("\nGeneration completed successfully!")

print(
    f"Total requirements processed: "
    f"{len(df_result)}"
)

df_result[
    [
        "requirement_id",
        "description",
        "user_story",
        "acceptance_criteria"
    ]
]

[AI Pipeline] Generating User Stories and Acceptance Criteria using Gemini...
Processed 1/5: REQ-001
Processed 2/5: REQ-002
Processed 3/5: REQ-003
Processed 4/5: REQ-004
Processed 5/5: REQ-005

Generation completed successfully!
Total requirements processed: 5


,requirement_id,description,user_story,acceptance_criteria
0,REQ-001,The biggest issue right now is that when a pur...,"As a Branch Manager, I want purchase requests ...",1. Given a purchase request from a branch exce...
1,REQ-002,People keep submitting requests without attach...,"As a Procurement Specialist, I want the system...",1. Given a user is filling out a purchase requ...
2,REQ-003,The system does not send any reminder if an ap...,"As an approver, I want to receive automated re...",1. Given an approval request has been pending ...
3,REQ-004,Need approvals to happen within 3 business day...,"As an approver, I want pending approval reques...","1. Given an approval request is submitted, Whe..."
4,REQ-005,Finance should see a simple dashboard of all p...,"As a finance team member, I want to view a sim...",1. Given a finance user is logged into the sys...


In [11]:
output_file = "user_stories_output.csv"

df_result.to_csv(
    output_file,
    index=False,
    encoding="utf-8"
)

print(
    f"Output file created successfully: "
    f"{output_file}"
)
files.download(output_file)

Output file created successfully: user_stories_output.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>